In [10]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

TRAINING_ROOT = Path(
    "/home/yuanzn/Documents/QUIC_Joint_Lifecycle_Scheduling/python/training_data"
)

RECEIVER_ROOT = Path(
    "/home/yuanzn/Documents/QUIC_Joint_Lifecycle_Scheduling/python/"
    "output_res/vvc_receiver"
)

VIDEO_LIST = [
    "BasketballDrive_QP32_3x3",
    "BQTerrace_QP32_3x3",
    "Cactus_QP32_3x3",
]

BW_LIST = [
    "bw0p1Mbps.npz",
    "bw0p5Mbps.npz",
    "bw1Mbps.npz",
    "bw2Mbps.npz",
    "bw5Mbps.npz",
]

FPS_MAP = {
    "BasketballDrive_QP32_3x3": 50.0,
    "BQTerrace_QP32_3x3": 60.0,
    "Cactus_QP32_3x3": 50.0,
}

RECEIVER_VIDEO_MAP = {
    "BasketballDrive_QP32_3x3": "BasketballDrive_1920x1080_50",
    "BQTerrace_QP32_3x3": "BQTerrace_1920x1080_60",
    "Cactus_QP32_3x3": "Cactus_1920x1080_50",
}

CONTROL_INTERVAL_S = 1.0


# ============================================================
# Helpers
# ============================================================

def build_frames_to_next_idr(nframes, idr_frames):
    idr_frames = np.asarray(idr_frames, dtype=np.int64)
    out = np.full(nframes, -1, dtype=np.int64)

    for fid in range(nframes):
        future = idr_frames[idr_frames >= fid]
        if len(future):
            out[fid] = int(future[0] - fid)

    return out


def bw_to_receiver_dir(bw_name):
    return bw_name.replace("bw", "")


def csv_to_matrix(df, col, nframes, num_subpics, dtype=np.float64):
    out = np.zeros((nframes, num_subpics), dtype=dtype)

    fid = df["frame_id"].to_numpy(np.int64)
    sid = df["subpic_id"].to_numpy(np.int64)
    val = pd.to_numeric(df[col], errors="coerce").fillna(0).to_numpy(dtype)

    valid = (
        (fid >= 0) & (fid < nframes) &
        (sid >= 0) & (sid < num_subpics)
    )

    out[fid[valid], sid[valid]] = val[valid]
    return out


# ============================================================
# Load + Normalize All Runs
# ============================================================

runs = {}

for video_id in VIDEO_LIST:
    video_dir = TRAINING_ROOT / video_id

    if not video_dir.exists():
        print(f"[WARN] Missing directory: {video_dir}")
        continue

    for bw_file in BW_LIST:
        npz_path = video_dir / bw_file

        if not npz_path.exists():
            print(f"[WARN] Missing NPZ: {npz_path}")
            continue

        # ----------------------------------------------------
        # Sender NPZ
        # ----------------------------------------------------

        with np.load(npz_path, allow_pickle=True) as data:
            raw = {k: data[k].copy() for k in data.files}

        bw_name = bw_file.replace(".npz", "")

        frame_id = np.asarray(raw["frame_id"], dtype=np.int64)
        roi = np.asarray(raw["roi"], dtype=np.float32)
        change = np.asarray(raw["change"], dtype=np.float32)
        frame_bytes = np.asarray(raw["frame_bytes"], dtype=np.float32)
        active = np.asarray(raw["active"], dtype=np.float32)

        rtt_us = np.asarray(raw["rtt_us"], dtype=np.float32)
        cwnd_bytes = np.asarray(raw["cwnd_bytes"], dtype=np.float32)
        bytes_in_flight = np.asarray(raw["bytes_in_flight"], dtype=np.float32)

        frame_source_time_ns = np.asarray(
            raw["frame_source_time_ns"],
            dtype=np.int64
        )

        subpic_send_time_ns = np.asarray(
            raw["subpic_send_time_ns"],
            dtype=np.int64
        )

        idr_frames = np.asarray(raw["idr_frames"], dtype=np.int64)
        is_idr = np.asarray(raw["is_idr"], dtype=np.int8)

        nframes = len(frame_id)
        num_subpics = roi.shape[1]

        assert roi.shape == (nframes, num_subpics)
        assert change.shape == (nframes, num_subpics)
        assert frame_bytes.shape == (nframes, num_subpics)
        assert active.shape == (nframes, num_subpics)
        assert subpic_send_time_ns.shape == (nframes, num_subpics)

        assert len(rtt_us) == nframes
        assert len(cwnd_bytes) == nframes
        assert len(bytes_in_flight) == nframes
        assert len(frame_source_time_ns) == nframes
        assert len(is_idr) == nframes

        frames_to_next_idr = build_frames_to_next_idr(
            nframes,
            idr_frames
        )

        # ----------------------------------------------------
        # Receiver CSV
        # ----------------------------------------------------

        receiver_video = RECEIVER_VIDEO_MAP[video_id]
        receiver_bw = bw_to_receiver_dir(bw_name)

        csv_path = (
            RECEIVER_ROOT
            / receiver_video
            / receiver_bw
            / "stream_timing.csv"
        )

        if not csv_path.exists():
            print(f"[WARN] Missing receiver CSV: {csv_path}")
            continue

        df = pd.read_csv(csv_path)

        receive_first_ns = csv_to_matrix(
            df, "receive_first_ns", nframes, num_subpics
        )

        receive_complete_ns = csv_to_matrix(
            df, "receive_complete_ns", nframes, num_subpics
        )

        frame_ready_ns = csv_to_matrix(
            df, "frame_ready_ns", nframes, num_subpics
        )

        scheduled_play_ns = csv_to_matrix(
            df, "scheduled_play_ns", nframes, num_subpics
        )

        actual_play_ns = csv_to_matrix(
            df, "actual_play_ns", nframes, num_subpics
        )

        decode_start_ns = csv_to_matrix(
            df, "decode_start_ns", nframes, num_subpics
        )

        decode_end_ns = csv_to_matrix(
            df, "decode_end_ns", nframes, num_subpics
        )

        received_complete = csv_to_matrix(
            df, "received_complete", nframes, num_subpics, np.float32
        )

        played = csv_to_matrix(
            df, "played", nframes, num_subpics, np.float32
        )

        feed_success = csv_to_matrix(
            df, "feed_success", nframes, num_subpics, np.float32
        )

        # ----------------------------------------------------
        # Store
        # ----------------------------------------------------

        runs[(video_id, bw_name)] = {
            "video_id": video_id,
            "bandwidth": bw_name,
            "fps": FPS_MAP[video_id],

            "frame_id": frame_id,
            "roi": roi,
            "change": change,
            "frame_bytes": frame_bytes,
            "active": active,

            "rtt_us": rtt_us,
            "cwnd_bytes": cwnd_bytes,
            "bytes_in_flight": bytes_in_flight,

            "frame_source_time_ns": frame_source_time_ns,
            "subpic_send_time_ns": subpic_send_time_ns,

            "receive_first_ns": receive_first_ns,
            "receive_complete_ns": receive_complete_ns,
            "frame_ready_ns": frame_ready_ns,
            "scheduled_play_ns": scheduled_play_ns,
            "actual_play_ns": actual_play_ns,
            "decode_start_ns": decode_start_ns,
            "decode_end_ns": decode_end_ns,

            "received_complete": received_complete,
            "played": played,
            "feed_success": feed_success,

            "idr_frames": idr_frames,
            "is_idr": is_idr,
            "frames_to_next_idr": frames_to_next_idr,

            "nframes": nframes,
            "num_subpics": num_subpics,
        }

        print(
            f"[LOAD] {video_id:30s} {bw_name:10s} "
            f"frames={nframes:4d} SPs={num_subpics} "
            f"recv_rows={len(df):5d} "
            f"IDR={idr_frames.tolist()}"
        )

print(f"\nTotal runs loaded: {len(runs)}")

[LOAD] BasketballDrive_QP32_3x3       bw0p1Mbps  frames= 500 SPs=9 recv_rows= 4500 IDR=[0, 32, 64, 96, 128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480]
[LOAD] BasketballDrive_QP32_3x3       bw0p5Mbps  frames= 500 SPs=9 recv_rows= 4500 IDR=[0, 32, 64, 96, 128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480]
[LOAD] BasketballDrive_QP32_3x3       bw1Mbps    frames= 500 SPs=9 recv_rows= 4500 IDR=[0, 32, 64, 96, 128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480]
[LOAD] BasketballDrive_QP32_3x3       bw2Mbps    frames= 500 SPs=9 recv_rows= 4500 IDR=[0, 32, 64, 96, 128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480]
[LOAD] BasketballDrive_QP32_3x3       bw5Mbps    frames= 500 SPs=9 recv_rows= 4500 IDR=[0, 32, 64, 96, 128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480]
[LOAD] BQTerrace_QP32_3x3             bw0p1Mbps  frames= 600 SPs=9 recv_rows= 5400 IDR=[0, 32, 64, 96, 128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480, 512, 544, 576]
[LOAD] 

In [12]:
# ============================================================
# Aggregate One Control Window
# ============================================================

def aggregate_window(run, start_fid, end_fid):
    sl = slice(start_fid, end_fid)
    decision_fid = end_fid - 1

    roi_w = run["roi"][sl]
    change_w = run["change"][sl]
    rtt_w = run["rtt_us"][sl]
    cwnd_w = run["cwnd_bytes"][sl]
    bif_w = run["bytes_in_flight"][sl]

    roi_mean = roi_w.mean(axis=0)
    roi_std = roi_w.std(axis=0)
    roi_last = roi_w[-1].copy()

    change_mean = change_w.mean(axis=0)
    change_std = change_w.std(axis=0)
    change_last = change_w[-1].copy()

    frame_bytes = run["frame_bytes"][sl].sum(axis=0)
    active = run["active"][decision_fid].copy()

    rtt_mean, rtt_std, rtt_last = rtt_w.mean(), rtt_w.std(), rtt_w[-1]
    cwnd_mean, cwnd_std, cwnd_last = cwnd_w.mean(), cwnd_w.std(), cwnd_w[-1]
    bif_mean, bif_std, bif_last = bif_w.mean(), bif_w.std(), bif_w[-1]

    bif_cwnd_mean = float(bif_mean / max(float(cwnd_mean), 1.0))
    bif_cwnd_last = float(bif_last / max(float(cwnd_last), 1.0))

    return {
        "video_id": run["video_id"],
        "bandwidth": run["bandwidth"],
        "fps": float(run["fps"]),
        "start_fid": int(start_fid),
        "end_fid": int(end_fid - 1),
        "decision_fid": int(decision_fid),

        "roi_mean": roi_mean.astype(np.float32),
        "roi_std": roi_std.astype(np.float32),
        "roi_last": roi_last.astype(np.float32),

        "change_mean": change_mean.astype(np.float32),
        "change_std": change_std.astype(np.float32),
        "change_last": change_last.astype(np.float32),

        "frame_bytes": frame_bytes.astype(np.float32),
        "active": active.astype(np.float32),

        "rtt_mean": float(rtt_mean),
        "rtt_std": float(rtt_std),
        "rtt_last": float(rtt_last),

        "cwnd_mean": float(cwnd_mean),
        "cwnd_std": float(cwnd_std),
        "cwnd_last": float(cwnd_last),

        "bif_mean": float(bif_mean),
        "bif_std": float(bif_std),
        "bif_last": float(bif_last),

        "bif_cwnd_mean": bif_cwnd_mean,
        "bif_cwnd_last": bif_cwnd_last,

        "frames_to_next_idr": int(run["frames_to_next_idr"][decision_fid]),
        "is_idr": int(run["is_idr"][decision_fid]),
    }


# ============================================================
# Build Samples for One Run
# ============================================================

def build_control_samples(run):
    control_frames = max(1, int(round(float(run["fps"]) * CONTROL_INTERVAL_S)))
    samples = []

    for start_fid in range(0, run["nframes"], control_frames):
        end_fid = min(start_fid + control_frames, run["nframes"])

        if end_fid - start_fid < control_frames:
            continue

        sample = aggregate_window(run, start_fid, end_fid)
        sample["control_idx"] = len(samples)
        samples.append(sample)

    return samples


# ============================================================
# Build All Samples
# ============================================================

samples_by_run = {}
all_samples = []

for run_key, run in runs.items():
    samples = build_control_samples(run)
    samples_by_run[run_key] = samples
    all_samples.extend(samples)

    print(
        f"[SAMPLE] {run['video_id']:30s} "
        f"{run['bandwidth']:12s} windows={len(samples)}"
    )

print(f"\nTotal control samples: {len(all_samples)}")


# ============================================================
# Train / Validation Split
# ============================================================

TRAIN_VIDEOS = {
    "BasketballDrive_QP32_3x3",
    "Cactus_QP32_3x3",
}

VAL_VIDEOS = {
    "BQTerrace_QP32_3x3",
}

train_runs = {
    key: samples
    for key, samples in samples_by_run.items()
    if key[0] in TRAIN_VIDEOS
}

val_runs = {
    key: samples
    for key, samples in samples_by_run.items()
    if key[0] in VAL_VIDEOS
}

train_samples = [s for samples in train_runs.values() for s in samples]
val_samples = [s for samples in val_runs.values() for s in samples]

print("\n" + "=" * 70)
print(f"Train runs       : {len(train_runs)}")
print(f"Train samples    : {len(train_samples)}")
print(f"Validation runs  : {len(val_runs)}")
print(f"Validation samples: {len(val_samples)}")


# ============================================================
# Quick Sample Check
# ============================================================

if all_samples:
    s = all_samples[0]

    print("\n" + "=" * 70)
    print("Example Sample")
    print("=" * 70)

    print(f"video_id   : {s['video_id']}")
    print(f"bandwidth  : {s['bandwidth']}")
    print(f"frame range: {s['start_fid']} ~ {s['end_fid']}")

    print("\nroi_mean   :", np.round(s["roi_mean"], 4))
    print("roi_std    :", np.round(s["roi_std"], 4))
    print("roi_last   :", np.round(s["roi_last"], 4))

    print("\nchange_mean:", np.round(s["change_mean"], 4))
    print("change_std :", np.round(s["change_std"], 4))
    print("change_last:", np.round(s["change_last"], 4))

    print("\nframe_bytes:", s["frame_bytes"].astype(np.int64))

    print(
        f"\nRTT       : mean={s['rtt_mean']/1000:.3f} ms, "
        f"std={s['rtt_std']/1000:.3f} ms, "
        f"last={s['rtt_last']/1000:.3f} ms"
    )

    print(
        f"CWND      : mean={s['cwnd_mean']:.1f} B, "
        f"std={s['cwnd_std']:.1f} B, "
        f"last={s['cwnd_last']:.1f} B"
    )

    print(
        f"BIF       : mean={s['bif_mean']:.1f} B, "
        f"std={s['bif_std']:.1f} B, "
        f"last={s['bif_last']:.1f} B"
    )

    print(
        f"BIF/CWND  : mean={s['bif_cwnd_mean']:.3f}, "
        f"last={s['bif_cwnd_last']:.3f}"
    )

    print(f"\nframes_to_next_idr: {s['frames_to_next_idr']}")
    print(f"is_idr            : {s['is_idr']}")

[SAMPLE] BasketballDrive_QP32_3x3       bw0p1Mbps    windows=10
[SAMPLE] BasketballDrive_QP32_3x3       bw0p5Mbps    windows=10
[SAMPLE] BasketballDrive_QP32_3x3       bw1Mbps      windows=10
[SAMPLE] BasketballDrive_QP32_3x3       bw2Mbps      windows=10
[SAMPLE] BasketballDrive_QP32_3x3       bw5Mbps      windows=10
[SAMPLE] BQTerrace_QP32_3x3             bw0p1Mbps    windows=10
[SAMPLE] BQTerrace_QP32_3x3             bw0p5Mbps    windows=10
[SAMPLE] BQTerrace_QP32_3x3             bw1Mbps      windows=10
[SAMPLE] BQTerrace_QP32_3x3             bw2Mbps      windows=10
[SAMPLE] BQTerrace_QP32_3x3             bw5Mbps      windows=10
[SAMPLE] Cactus_QP32_3x3                bw0p1Mbps    windows=10
[SAMPLE] Cactus_QP32_3x3                bw0p5Mbps    windows=10
[SAMPLE] Cactus_QP32_3x3                bw1Mbps      windows=10
[SAMPLE] Cactus_QP32_3x3                bw2Mbps      windows=10
[SAMPLE] Cactus_QP32_3x3                bw5Mbps      windows=10

Total control samples: 150

Train runs 

In [27]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Beta

# ============================================================
# Config
# ============================================================

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPS = 1e-8

ACTIVE, CLOSED, PENDING_REOPEN = 1, 0, 2

LR = 3e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
PPO_CLIP = 0.2
VALUE_COEF = 0.5
ENTROPY_COEF = 0.01
MAX_GRAD_NORM = 0.5

PPO_EPOCHS = 6
MINIBATCH_SIZE = 64
NUM_UPDATES = 200

SWITCH_WEIGHT = 0.05

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("DEVICE:", DEVICE)


# ============================================================
# Basic Utilities
# ============================================================

def normalize_unit(x):
    x = np.asarray(x, dtype=np.float32)
    s = float(x.sum())
    return np.zeros_like(x) if s <= EPS else x / s


def next_idr_after(run, fid):
    ids = np.asarray(run["idr_frames"], dtype=np.int64)
    future = ids[ids >= fid]
    return None if len(future) == 0 else int(future[0])


# ============================================================
# Lifecycle Selection
# ============================================================

def lifecycle_target(sample, rho, lambda_):
    rho = float(np.clip(rho, 0, 1))
    lambda_ = float(np.clip(lambda_, 0, 1))

    R = np.asarray(sample["roi_mean"], dtype=np.float32)
    T = np.asarray(sample["change_mean"], dtype=np.float32)
    frame_bytes = np.asarray(sample["frame_bytes"], dtype=np.float32)

    R_hat = normalize_unit(R)
    T_hat = normalize_unit(T)
    score = lambda_ * R_hat + (1 - lambda_) * T_hat

    total_bytes = float(frame_bytes.sum())
    target_bytes = rho * total_bytes

    order = np.argsort(-score, kind="stable")
    selected, used_bytes = [], 0.0

    for sid in order:
        sid = int(sid)
        selected.append(sid)
        used_bytes += float(frame_bytes[sid])
        if used_bytes >= target_bytes:
            break

    if not selected and len(order):
        sid = int(order[0])
        selected = [sid]
        used_bytes = float(frame_bytes[sid])

    return {
        "selected": selected,
        "score": score.astype(np.float32),
        "R_hat": R_hat.astype(np.float32),
        "T_hat": T_hat.astype(np.float32),
        "target_bytes": target_bytes,
        "estimated_bytes": used_bytes,
        "total_bytes": total_bytes,
    }


# ============================================================
# PPO Observation
# ============================================================

def raw_global_features(sample):
    fps = float(sample["fps"])
    next_idr = int(sample["frames_to_next_idr"])
    has_idr = float(next_idr >= 0)
    idr_wait_s = max(next_idr, 0) / max(fps, EPS)

    return np.asarray([
        np.mean(sample["roi_mean"]),
        np.mean(sample["change_mean"]),
        np.mean(sample["roi_std"]),
        np.mean(sample["change_std"]),

        sample["rtt_mean"] / 1000.0,
        sample["rtt_std"] / 1000.0,
        sample["rtt_last"] / 1000.0,

        np.log1p(sample["cwnd_mean"]),
        np.log1p(sample["cwnd_std"]),
        np.log1p(sample["cwnd_last"]),

        np.log1p(sample["bif_mean"]),
        np.log1p(sample["bif_std"]),
        np.log1p(sample["bif_last"]),

        sample["bif_cwnd_mean"],
        sample["bif_cwnd_last"],

        idr_wait_s,
        has_idr,
        float(sample["is_idr"]),
    ], dtype=np.float32)


def fit_global_scaler(run_dict):
    X = np.stack([raw_global_features(s) for samples in run_dict.values() for s in samples])
    mean, std = X.mean(axis=0), X.std(axis=0)
    std[std < 1e-6] = 1.0
    return mean.astype(np.float32), std.astype(np.float32)


GLOBAL_MEAN, GLOBAL_STD = fit_global_scaler(train_runs)
GLOBAL_DIM = len(GLOBAL_MEAN)


def normalize_global(sample):
    x = raw_global_features(sample)
    return ((x - GLOBAL_MEAN) / GLOBAL_STD).astype(np.float32)


def build_sp_features(sample, lifecycle_state):
    R = np.asarray(sample["roi_mean"], dtype=np.float32)
    T = np.asarray(sample["change_mean"], dtype=np.float32)
    R_std = np.asarray(sample["roi_std"], dtype=np.float32)
    T_std = np.asarray(sample["change_std"], dtype=np.float32)
    bytes_ = np.asarray(sample["frame_bytes"], dtype=np.float32)

    R_hat = normalize_unit(R)
    T_hat = normalize_unit(T)
    byte_ratio = bytes_ / max(float(bytes_.sum()), EPS)

    active = (lifecycle_state == ACTIVE).astype(np.float32)
    pending = (lifecycle_state == PENDING_REOPEN).astype(np.float32)

    return np.stack([
        R, T, R_std, T_std,
        R_hat, T_hat, byte_ratio,
        active, pending
    ], axis=1).astype(np.float32)


SP_DIM = 9


# ============================================================
# Actor-Critic
# ============================================================

class LifecycleActorCritic(nn.Module):
    def __init__(self, sp_dim=SP_DIM, global_dim=GLOBAL_DIM, hidden=64):
        super().__init__()

        self.sp_encoder = nn.Sequential(
            nn.Linear(sp_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU()
        )

        self.global_encoder = nn.Sequential(
            nn.Linear(global_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU()
        )

        self.fusion = nn.Sequential(
            nn.Linear(hidden * 3, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU()
        )

        self.actor_head = nn.Linear(64, 4)
        self.critic_head = nn.Linear(64, 1)

    def forward(self, sp_x, global_x):
        h = self.sp_encoder(sp_x)
        h_mean = h.mean(dim=0)
        h_max = h.max(dim=0).values
        g = self.global_encoder(global_x)

        z = self.fusion(torch.cat([h_mean, h_max, g], dim=-1))
        params = F.softplus(self.actor_head(z)) + 1.0

        dist_rho = Beta(params[0], params[1])
        dist_lambda = Beta(params[2], params[3])
        value = self.critic_head(z).squeeze(-1)

        return dist_rho, dist_lambda, value


model = LifecycleActorCritic().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(model)


# ============================================================
# Environment
# ============================================================

class LifecycleEnv:
    def __init__(self, samples, run):
        self.samples = samples
        self.run = run
        self.num_subpics = int(run["num_subpics"])
        self.reset()

    def reset(self):
        self.t = 0
        self.state = np.full(self.num_subpics, ACTIVE, dtype=np.int8)
        self.pending_fid = np.full(self.num_subpics, -1, dtype=np.int64)
        return self.get_obs()

    def get_obs(self):
        sample = self.samples[self.t]
        return build_sp_features(sample, self.state), normalize_global(sample)

    def network_pressure(self, sample):
        x = max(float(sample["bif_cwnd_mean"]), float(sample["bif_cwnd_last"]))
        return float(np.clip(x, 0, 1))

    def simulate_next_window(self, target_active, next_sample):
        target_active = set(map(int, target_active))
        next_start = int(next_sample["start_fid"])
        close_count = 0

        for sid in range(self.num_subpics):
            want_active = sid in target_active

            if self.state[sid] == ACTIVE and not want_active:
                self.state[sid] = CLOSED
                self.pending_fid[sid] = -1
                close_count += 1

            elif self.state[sid] == CLOSED and want_active:
                next_idr = next_idr_after(self.run, next_start)
                if next_idr is not None:
                    self.state[sid] = PENDING_REOPEN
                    self.pending_fid[sid] = next_idr

            elif self.state[sid] == PENDING_REOPEN and not want_active:
                self.state[sid] = CLOSED
                self.pending_fid[sid] = -1

        start, end = int(next_sample["start_fid"]), int(next_sample["end_fid"])
        total_bytes = sent_bytes = total_roi = kept_roi = 0.0
        reopen_count = 0

        for fid in range(start, end + 1):
            for sid in range(self.num_subpics):
                if self.state[sid] == PENDING_REOPEN and self.pending_fid[sid] == fid:
                    self.state[sid] = ACTIVE
                    self.pending_fid[sid] = -1
                    reopen_count += 1

            bytes_f = np.asarray(self.run["frame_bytes"][fid], dtype=np.float32)
            roi_f = np.asarray(self.run["roi"][fid], dtype=np.float32)
            active_mask = self.state == ACTIVE

            total_bytes += float(bytes_f.sum())
            sent_bytes += float(bytes_f[active_mask].sum())
            total_roi += float(roi_f.sum())
            kept_roi += float(roi_f[active_mask].sum())

        actual_ratio = sent_bytes / max(total_bytes, EPS)
        semantic_keep = kept_roi / total_roi if total_roi > EPS else 1.0
        switch_ratio = (close_count + reopen_count) / max(self.num_subpics, 1)

        return {
            "sent_bytes": sent_bytes,
            "total_bytes": total_bytes,
            "actual_ratio": actual_ratio,
            "semantic_keep": semantic_keep,
            "semantic_loss": 1.0 - semantic_keep,
            "close_count": close_count,
            "reopen_count": reopen_count,
            "switch_ratio": switch_ratio,
        }

    def step(self, action):
        rho, lambda_ = map(float, action)
        current_sample = self.samples[self.t]

        decision = lifecycle_target(current_sample, rho, lambda_)
        target_active = decision["selected"]

        next_t = self.t + 1
        next_sample = self.samples[next_t]

        metrics = self.simulate_next_window(target_active, next_sample)
        pressure = self.network_pressure(current_sample)

        network_cost = pressure * metrics["actual_ratio"]
        reward = (
            metrics["semantic_keep"]
            - network_cost
            - SWITCH_WEIGHT * metrics["switch_ratio"]
        )

        info = {
            "rho": rho,
            "lambda": lambda_,
            "pressure": pressure,
            "target_active": target_active,
            "actual_active": np.where(self.state == ACTIVE)[0].tolist(),
            "reward": float(reward),
            **metrics
        }

        self.t = next_t
        done = self.t >= len(self.samples) - 1
        next_obs = None if done else self.get_obs()

        return next_obs, float(reward), done, info


# ============================================================
# PPO Helpers
# ============================================================

def obs_to_tensor(obs):
    sp, g = obs
    return (
        torch.tensor(sp, dtype=torch.float32, device=DEVICE),
        torch.tensor(g, dtype=torch.float32, device=DEVICE)
    )


@torch.no_grad()
def sample_action(model, obs, deterministic=False):
    sp, g = obs_to_tensor(obs)
    dist_rho, dist_lambda, value = model(sp, g)

    if deterministic:
        rho = dist_rho.concentration1 / (dist_rho.concentration1 + dist_rho.concentration0)
        lam = dist_lambda.concentration1 / (dist_lambda.concentration1 + dist_lambda.concentration0)
    else:
        rho = dist_rho.sample()
        lam = dist_lambda.sample()

    action = torch.stack([rho, lam])
    log_prob = dist_rho.log_prob(rho) + dist_lambda.log_prob(lam)

    return action.cpu().numpy(), float(log_prob.item()), float(value.item())


def collect_episode(model, samples, run):
    if len(samples) < 2:
        return []

    env = LifecycleEnv(samples, run)
    obs = env.reset()
    trajectory = []

    while True:
        action, log_prob, value = sample_action(model, obs, deterministic=False)
        next_obs, reward, done, info = env.step(action)

        sp, g = obs
        trajectory.append({
            "sp": sp.copy(),
            "global": g.copy(),
            "action": action.astype(np.float32),
            "old_log_prob": log_prob,
            "value": value,
            "reward": reward,
            "done": done,
            "info": info,
        })

        if done:
            break

        obs = next_obs

    return trajectory


def add_gae(traj):
    gae, next_value = 0.0, 0.0

    for t in reversed(range(len(traj))):
        reward = traj[t]["reward"]
        value = traj[t]["value"]
        mask = 0.0 if traj[t]["done"] else 1.0

        delta = reward + GAMMA * next_value * mask - value
        gae = delta + GAMMA * GAE_LAMBDA * mask * gae

        traj[t]["advantage"] = gae
        traj[t]["return"] = gae + value
        next_value = value

    return traj


# ============================================================
# PPO Update
# ============================================================

def ppo_update(model, optimizer, transitions):
    if not transitions:
        return {}

    adv = np.asarray([x["advantage"] for x in transitions], dtype=np.float32)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)

    for i, a in enumerate(adv):
        transitions[i]["advantage_norm"] = float(a)

    indices = np.arange(len(transitions))
    losses = []

    for _ in range(PPO_EPOCHS):
        np.random.shuffle(indices)

        for start in range(0, len(indices), MINIBATCH_SIZE):
            batch_ids = indices[start:start + MINIBATCH_SIZE]

            policy_losses, value_losses, entropies = [], [], []

            for idx in batch_ids:
                tr = transitions[int(idx)]

                sp = torch.tensor(tr["sp"], dtype=torch.float32, device=DEVICE)
                g = torch.tensor(tr["global"], dtype=torch.float32, device=DEVICE)
                action = torch.tensor(tr["action"], dtype=torch.float32, device=DEVICE)
                old_log_prob = torch.tensor(tr["old_log_prob"], dtype=torch.float32, device=DEVICE)
                advantage = torch.tensor(tr["advantage_norm"], dtype=torch.float32, device=DEVICE)
                target_return = torch.tensor(tr["return"], dtype=torch.float32, device=DEVICE)

                dist_rho, dist_lambda, value = model(sp, g)

                new_log_prob = (
                    dist_rho.log_prob(action[0])
                    + dist_lambda.log_prob(action[1])
                )

                entropy = dist_rho.entropy() + dist_lambda.entropy()
                ratio = torch.exp(new_log_prob - old_log_prob)

                unclipped = ratio * advantage
                clipped = torch.clamp(ratio, 1 - PPO_CLIP, 1 + PPO_CLIP) * advantage

                policy_losses.append(-torch.min(unclipped, clipped))
                value_losses.append((value - target_return).pow(2))
                entropies.append(entropy)

            policy_loss = torch.stack(policy_losses).mean()
            value_loss = torch.stack(value_losses).mean()
            entropy = torch.stack(entropies).mean()

            loss = policy_loss + VALUE_COEF * value_loss - ENTROPY_COEF * entropy

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()

            losses.append([
                float(loss.item()),
                float(policy_loss.item()),
                float(value_loss.item()),
                float(entropy.item())
            ])

    m = np.mean(losses, axis=0)
    return {"loss": m[0], "policy_loss": m[1], "value_loss": m[2], "entropy": m[3]}


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate_runs(model, run_dict, runs):
    model.eval()
    rows = []

    for run_key, samples in run_dict.items():
        if len(samples) < 2:
            continue

        env = LifecycleEnv(samples, runs[run_key])
        obs = env.reset()

        while True:
            action, _, _ = sample_action(model, obs, deterministic=True)
            next_obs, reward, done, info = env.step(action)

            rows.append({
                "video_id": run_key[0],
                "bandwidth": run_key[1],
                "rho": info["rho"],
                "lambda": info["lambda"],
                "pressure": info["pressure"],
                "reward": reward,
                "actual_ratio": info["actual_ratio"],
                "semantic_keep": info["semantic_keep"],
                "semantic_loss": info["semantic_loss"],
                "switch_ratio": info["switch_ratio"],
                "close_count": info["close_count"],
                "reopen_count": info["reopen_count"],
            })

            if done:
                break

            obs = next_obs

    return pd.DataFrame(rows)


# ============================================================
# Training
# ============================================================

history = []
train_keys = list(train_runs.keys())
best_val_reward = -np.inf
best_state = None

for update in range(1, NUM_UPDATES + 1):
    model.train()
    random.shuffle(train_keys)

    transitions = []
    episode_rewards = []

    for run_key in train_keys:
        traj = collect_episode(model, train_runs[run_key], runs[run_key])
        traj = add_gae(traj)

        transitions.extend(traj)

        if traj:
            episode_rewards.append(np.mean([x["reward"] for x in traj]))

    loss_info = ppo_update(model, optimizer, transitions)

    val_df = evaluate_runs(model, val_runs, runs)
    val_reward = val_df["reward"].mean() if len(val_df) else np.nan
    val_rho = val_df["rho"].mean() if len(val_df) else np.nan
    val_lambda = val_df["lambda"].mean() if len(val_df) else np.nan

    row = {
        "update": update,
        "train_reward": float(np.mean(episode_rewards)),
        "val_reward": float(val_reward),
        "val_rho": float(val_rho),
        "val_lambda": float(val_lambda),
        **loss_info
    }

    history.append(row)

    if np.isfinite(val_reward) and val_reward > best_val_reward:
        best_val_reward = val_reward
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if update == 1 or update % 10 == 0:
        print(
            f"[{update:03d}/{NUM_UPDATES}] "
            f"train={row['train_reward']:.4f} "
            f"val={row['val_reward']:.4f} "
            f"rho={row['val_rho']:.3f} "
            f"lambda={row['val_lambda']:.3f} "
            f"loss={row.get('loss', np.nan):.4f}"
        )


# ============================================================
# Restore Best Model
# ============================================================

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nLoaded best model | val_reward={best_val_reward:.4f}")


# ============================================================
# Final Results
# ============================================================

history_df = pd.DataFrame(history)
display(history_df.tail(20))

train_eval = evaluate_runs(model, train_runs, runs)
val_eval = evaluate_runs(model, val_runs, runs)


def summarize_eval(df, name):
    print(f"\n{'=' * 80}\n{name}\n{'=' * 80}")

    summary = (
        df.groupby("bandwidth")
        .agg(
            reward=("reward", "mean"),
            rho=("rho", "mean"),
            lambda_=("lambda", "mean"),
            pressure=("pressure", "mean"),
            actual_ratio=("actual_ratio", "mean"),
            semantic_keep=("semantic_keep", "mean"),
            semantic_loss=("semantic_loss", "mean"),
            switch_ratio=("switch_ratio", "mean"),
        )
        .reset_index()
    )

    display(summary)
    return summary


train_summary = summarize_eval(train_eval, "TRAIN RESULTS")
val_summary = summarize_eval(val_eval, "VALIDATION RESULTS")


val_action_summary = (
    val_eval.groupby(["video_id", "bandwidth"])
    .agg(
        rho_mean=("rho", "mean"),
        rho_std=("rho", "std"),
        lambda_mean=("lambda", "mean"),
        lambda_std=("lambda", "std"),
        pressure=("pressure", "mean"),
        actual_ratio=("actual_ratio", "mean"),
        semantic_keep=("semantic_keep", "mean"),
        reward=("reward", "mean"),
    )
    .reset_index()
)

print("\nVALIDATION LEARNED ACTIONS")
display(val_action_summary)

DEVICE: cuda
LifecycleActorCritic(
  (sp_encoder): Sequential(
    (0): Linear(in_features=9, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
  )
  (global_encoder): Sequential(
    (0): Linear(in_features=18, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
  )
  (fusion): Sequential(
    (0): Linear(in_features=192, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
  )
  (actor_head): Linear(in_features=64, out_features=4, bias=True)
  (critic_head): Linear(in_features=64, out_features=1, bias=True)
)
[001/200] train=0.1986 val=0.3678 rho=0.496 lambda=0.503 loss=0.8984
[010/200] train=0.2186 val=0.3846 rho=0.504 lambda=0.536 loss=0.3143
[020/200] train=0.2725 val=0.3973 rho=0.530 lambda=0.571 loss=0.4511
[030/200] train=0.3165 val=0.4185 rho=0.602 lambda=0.703 loss=0.2412


,update,train_reward,val_reward,val_rho,val_lambda,loss,policy_loss,value_loss,entropy
180,181,0.426945,0.378824,0.817412,0.924942,0.009076,-0.031072,0.019606,-3.034543
181,182,0.435990,0.365312,0.840520,0.922966,0.035164,-0.010062,0.028642,-3.090494
182,183,0.428003,0.364501,0.840859,0.928402,0.037761,-0.007893,0.028713,-3.129799
183,184,0.432117,0.370648,0.839986,0.926300,0.001343,-0.043207,0.025202,-3.194907
184,185,0.428903,0.373814,0.826769,0.923170,0.046695,0.003080,0.024353,-3.143911
185,186,0.431972,0.381204,0.808324,0.922577,0.031749,-0.010270,0.021793,-3.112226
186,187,0.428034,0.379754,0.818085,0.914557,0.019186,-0.023597,0.024757,-3.040440
187,188,0.440717,0.380358,0.814119,0.908817,0.022641,-0.019839,0.025651,-2.965506
188,189,0.429534,0.380358,0.816547,0.900796,0.013799,-0.023362,0.016371,-2.897590
189,190,0.422694,0.375655,0.829200,0.899133,0.011322,-0.034973,0.035294,-2.864811



TRAIN RESULTS


,bandwidth,reward,rho,lambda_,pressure,actual_ratio,semantic_keep,semantic_loss,switch_ratio
0,bw0p1Mbps,0.244627,0.470479,0.922992,0.999995,0.503938,0.757205,0.242795,0.172840
1,bw0p5Mbps,0.227020,0.429726,0.919878,0.999256,0.451689,0.687640,0.312360,0.185185
2,bw1Mbps,0.244625,0.472842,0.919523,1.000000,0.503938,0.757205,0.242795,0.172840
3,bw2Mbps,0.486214,0.549663,0.918229,0.557314,0.563611,0.794579,0.205421,0.172840
4,bw5Mbps,0.870608,0.716329,0.920311,0.109499,0.758880,0.958268,0.041732,0.086420



VALIDATION RESULTS


,bandwidth,reward,rho,lambda_,pressure,actual_ratio,semantic_keep,semantic_loss,switch_ratio
0,bw0p1Mbps,0.230878,0.556456,0.917345,1.000000,0.612910,0.853664,0.146336,0.197531
1,bw0p5Mbps,0.220549,0.618148,0.912368,1.000000,0.662614,0.893040,0.106960,0.197531
2,bw1Mbps,0.220549,0.602021,0.913076,1.000000,0.662614,0.893040,0.106960,0.197531
3,bw2Mbps,0.692732,0.648968,0.913282,0.342881,0.721699,0.928096,0.071904,0.148148
4,bw5Mbps,0.822508,0.665277,0.915579,0.125510,0.724810,0.917952,0.082048,0.148148



VALIDATION LEARNED ACTIONS


,video_id,bandwidth,rho_mean,rho_std,lambda_mean,lambda_std,pressure,actual_ratio,semantic_keep,reward
0,BQTerrace_QP32_3x3,bw0p1Mbps,0.556456,0.035398,0.917345,0.003041,1.000000,0.612910,0.853664,0.230878
1,BQTerrace_QP32_3x3,bw0p5Mbps,0.618148,0.055075,0.912368,0.001456,1.000000,0.662614,0.893040,0.220549
2,BQTerrace_QP32_3x3,bw1Mbps,0.602021,0.029701,0.913076,0.001524,1.000000,0.662614,0.893040,0.220549
3,BQTerrace_QP32_3x3,bw2Mbps,0.648968,0.050278,0.913282,0.002108,0.342881,0.721699,0.928096,0.692732
4,BQTerrace_QP32_3x3,bw5Mbps,0.665277,0.044373,0.915579,0.002231,0.125510,0.724810,0.917952,0.822508


In [28]:
# ============================================================
# Lifecycle Policy Debug
# ============================================================

import numpy as np
import pandas as pd
import torch

EPS = 1e-8


# ============================================================
# Build Active History
# ============================================================

def simulate_lifecycle_window(run, state, pending, target, start, end):
    nsp = int(run["num_subpics"])
    idrs = np.asarray(run["idr_frames"], dtype=np.int64)

    close_count = 0
    reopen_count = 0

    # --------------------------------------------------------
    # Lifecycle decision at control boundary
    # --------------------------------------------------------
    for sid in range(nsp):
        want = sid in target

        if state[sid] == ACTIVE and not want:
            state[sid] = CLOSED
            pending[sid] = -1
            close_count += 1

        elif state[sid] == CLOSED and want:
            future = idrs[idrs >= start]

            if len(future):
                state[sid] = PENDING_REOPEN
                pending[sid] = int(future[0])

        elif state[sid] == PENDING_REOPEN and not want:
            state[sid] = CLOSED
            pending[sid] = -1

    # --------------------------------------------------------
    # Frame-level state evolution
    # --------------------------------------------------------
    active_history = []

    for fid in range(start, end + 1):
        for sid in range(nsp):
            if (
                state[sid] == PENDING_REOPEN
                and pending[sid] == fid
            ):
                state[sid] = ACTIVE
                pending[sid] = -1
                reopen_count += 1

        active_history.append(
            (state == ACTIVE).copy()
        )

    return active_history, close_count, reopen_count


# ============================================================
# Window Statistics
# ============================================================

def calc_window_stats(run, start, end, active_history):
    total_bytes = 0.0
    sent_bytes = 0.0

    total_roi = 0.0
    kept_roi = 0.0

    active_ratios = []

    for k, fid in enumerate(range(start, end + 1)):
        active = np.asarray(
            active_history[k],
            dtype=bool
        )

        B = np.asarray(
            run["frame_bytes"][fid],
            dtype=np.float64
        )

        R = np.asarray(
            run["roi"][fid],
            dtype=np.float64
        )

        total_bytes += float(B.sum())
        sent_bytes += float(B[active].sum())

        total_roi += float(R.sum())
        kept_roi += float(R[active].sum())

        active_ratios.append(
            active.mean()
        )

    actual_ratio = sent_bytes / max(total_bytes, EPS)
    saved_ratio = 1.0 - actual_ratio

    semantic_keep = (
        kept_roi / max(total_roi, EPS)
        if total_roi > EPS
        else 1.0
    )

    return {
        "actual_ratio": actual_ratio,
        "saved_ratio": saved_ratio,
        "semantic_keep": semantic_keep,
        "active_ratio": float(np.mean(active_ratios)),
    }


# ============================================================
# Run PPO Lifecycle Debug
# ============================================================

@torch.no_grad()
def debug_lifecycle(model, run_dict, runs):
    model.eval()
    rows = []

    for run_key, samples in run_dict.items():
        if len(samples) < 2:
            continue

        video_id, bandwidth = run_key
        run = runs[run_key]

        nsp = int(run["num_subpics"])

        state = np.full(
            nsp,
            ACTIVE,
            dtype=np.int8
        )

        pending = np.full(
            nsp,
            -1,
            dtype=np.int64
        )

        for t in range(len(samples) - 1):
            s = samples[t]
            nxt = samples[t + 1]

            # ------------------------------------------------
            # PPO action
            # ------------------------------------------------
            obs = (
                build_sp_features(s, state),
                normalize_global(s)
            )

            action, _, _ = sample_action(
                model,
                obs,
                deterministic=True
            )

            rho, lam = map(float, action)

            decision = lifecycle_target(
                s,
                rho,
                lam
            )

            target = set(
                map(
                    int,
                    decision["selected"]
                )
            )

            start = int(nxt["start_fid"])
            end = int(nxt["end_fid"])

            # ------------------------------------------------
            # State before control action
            # ------------------------------------------------
            before_state = state.copy()

            before_active = tuple(
                np.flatnonzero(
                    before_state == ACTIVE
                ).tolist()
            )

            before_closed = tuple(
                np.flatnonzero(
                    before_state == CLOSED
                ).tolist()
            )

            before_pending = tuple(
                np.flatnonzero(
                    before_state == PENDING_REOPEN
                ).tolist()
            )

            # ------------------------------------------------
            # Simulate lifecycle
            # ------------------------------------------------
            active_history, closes, reopens = simulate_lifecycle_window(
                run,
                state,
                pending,
                target,
                start,
                end
            )

            after_state = state.copy()

            after_active = tuple(
                np.flatnonzero(
                    after_state == ACTIVE
                ).tolist()
            )

            after_closed = tuple(
                np.flatnonzero(
                    after_state == CLOSED
                ).tolist()
            )

            after_pending = tuple(
                np.flatnonzero(
                    after_state == PENDING_REOPEN
                ).tolist()
            )

            # ------------------------------------------------
            # Actual traffic / semantic effect
            # ------------------------------------------------
            stats = calc_window_stats(
                run,
                start,
                end,
                active_history
            )

            switch_count = closes + reopens
            switch_ratio = switch_count / max(nsp, 1)

            # ------------------------------------------------
            # Selector diagnostics
            # ------------------------------------------------
            target_tuple = tuple(
                sorted(target)
            )

            target_count = len(target)

            target_ratio = (
                target_count / nsp
            )

            rows.append({
                "video_id": video_id,
                "bandwidth": bandwidth,
                "control_idx": t,

                "start_fid": start,
                "end_fid": end,

                "rho": rho,
                "lambda": lam,

                "target": target_tuple,
                "target_count": target_count,
                "target_sp_ratio": target_ratio,

                "before_active": before_active,
                "before_closed": before_closed,
                "before_pending": before_pending,

                "after_active": after_active,
                "after_closed": after_closed,
                "after_pending": after_pending,

                "close_count": closes,
                "reopen_count": reopens,
                "switch_count": switch_count,
                "switch_ratio": switch_ratio,

                "actual_ratio": stats["actual_ratio"],
                "saved_ratio": stats["saved_ratio"],
                "semantic_keep": stats["semantic_keep"],
                "active_ratio": stats["active_ratio"],
            })

    return pd.DataFrame(rows)


# ============================================================
# Run Debug
# ============================================================

debug_df = debug_lifecycle(
    model,
    val_runs,
    runs
)


# ============================================================
# Detailed Control-Level Table
# ============================================================

detail_cols = [
    "bandwidth",
    "control_idx",

    "rho",
    "lambda",

    "target",
    "target_count",

    "before_active",
    "after_active",

    "close_count",
    "reopen_count",
    "switch_ratio",

    "actual_ratio",
    "saved_ratio",
    "semantic_keep",
]

print("\n" + "=" * 120)
print("CONTROL-LEVEL LIFECYCLE DEBUG")
print("=" * 120)

display(
    debug_df[detail_cols]
    .round({
        "rho": 4,
        "lambda": 4,
        "switch_ratio": 4,
        "actual_ratio": 4,
        "saved_ratio": 4,
        "semantic_keep": 4,
    })
)


# ============================================================
# Per-Bandwidth Summary
# ============================================================

summary = (
    debug_df
    .groupby("bandwidth")
    .agg(
        rho_mean=("rho", "mean"),
        rho_std=("rho", "std"),

        lambda_mean=("lambda", "mean"),
        lambda_std=("lambda", "std"),

        unique_targets=("target", "nunique"),
        target_count_mean=("target_count", "mean"),
        target_count_std=("target_count", "std"),

        close_mean=("close_count", "mean"),
        reopen_mean=("reopen_count", "mean"),

        switch_mean=("switch_ratio", "mean"),
        switch_std=("switch_ratio", "std"),

        actual_ratio_mean=("actual_ratio", "mean"),
        actual_ratio_std=("actual_ratio", "std"),

        saved_ratio_mean=("saved_ratio", "mean"),
        saved_ratio_std=("saved_ratio", "std"),

        semantic_keep_mean=("semantic_keep", "mean"),
        semantic_keep_std=("semantic_keep", "std"),

        active_ratio_mean=("active_ratio", "mean"),
    )
    .reset_index()
)

print("\n" + "=" * 120)
print("PER-BANDWIDTH LIFECYCLE SUMMARY")
print("=" * 120)

display(
    summary.round(4)
)


# ============================================================
# Target Frequency
# ============================================================

target_frequency = (
    debug_df
    .groupby(
        ["bandwidth", "target"]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        ["bandwidth", "count"],
        ascending=[True, False]
    )
)

print("\n" + "=" * 120)
print("TARGET-SET FREQUENCY")
print("=" * 120)

display(target_frequency)


# ============================================================
# Compact Sanity Check
# ============================================================

print("\n" + "=" * 120)
print("SANITY CHECK")
print("=" * 120)

for bw, g in debug_df.groupby("bandwidth"):
    print(
        f"{bw:10s} | "
        f"rho={g['rho'].mean():.4f}±{g['rho'].std():.4f} | "
        f"lambda={g['lambda'].mean():.4f}±{g['lambda'].std():.4f} | "
        f"unique_targets={g['target'].nunique():2d} | "
        f"target_n={g['target_count'].mean():.2f} | "
        f"saved={g['saved_ratio'].mean():.4f} | "
        f"switch={g['switch_ratio'].mean():.4f}"
    )


CONTROL-LEVEL LIFECYCLE DEBUG


,bandwidth,control_idx,rho,lambda,target,target_count,before_active,after_active,close_count,reopen_count,switch_ratio,actual_ratio,saved_ratio,semantic_keep
0,bw0p1Mbps,0,0.6412,0.9152,"(1, 3, 4, 5, 6, 7)",6,"(0, 1, 2, 3, 4, 5, 6, 7, 8)","(1, 3, 4, 5, 6, 7)",3,0,0.3333,0.7143,0.2857,0.9656
1,bw0p1Mbps,1,0.5681,0.9169,"(1, 3, 4, 6, 7)",5,"(1, 3, 4, 5, 6, 7)","(1, 3, 4, 6, 7)",1,0,0.1111,0.6470,0.3530,0.9818
2,bw0p1Mbps,2,0.5573,0.9137,"(1, 3, 4, 6, 7)",5,"(1, 3, 4, 6, 7)","(1, 3, 4, 6, 7)",0,0,0.0000,0.6303,0.3697,0.8573
3,bw0p1Mbps,3,0.5626,0.9155,"(0, 1, 3, 6, 7)",5,"(1, 3, 4, 6, 7)","(0, 1, 3, 6, 7)",1,1,0.2222,0.5967,0.4033,0.6988
4,bw0p1Mbps,4,0.5481,0.9171,"(0, 1, 2, 3)",4,"(0, 1, 3, 6, 7)","(0, 1, 2, 3)",2,1,0.3333,0.5496,0.4504,0.6614
5,bw0p1Mbps,5,0.5319,0.9174,"(0, 2, 3, 4, 6)",5,"(0, 1, 2, 3)","(0, 2, 3, 4, 6)",1,2,0.3333,0.5563,0.4437,0.7449
6,bw0p1Mbps,6,0.5245,0.9180,"(0, 3, 4, 5, 6)",5,"(0, 2, 3, 4, 6)","(0, 3, 4, 5, 6)",1,1,0.2222,0.5721,0.4279,0.8523
7,bw0p1Mbps,7,0.5263,0.9246,"(2, 3, 4, 5, 6)",5,"(0, 3, 4, 5, 6)","(2, 3, 4, 5, 6)",1,1,0.2222,0.6261,0.3739,0.9714
8,bw0p1Mbps,8,0.5480,0.9178,"(2, 3, 4, 5, 6)",5,"(2, 3, 4, 5, 6)","(2, 3, 4, 5, 6)",0,0,0.0000,0.6236,0.3764,0.9495
9,bw0p5Mbps,0,0.5858,0.9133,"(1, 3, 4, 5, 6)",5,"(0, 1, 2, 3, 4, 5, 6, 7, 8)","(1, 3, 4, 5, 6)",4,0,0.4444,0.5960,0.4040,0.9201



PER-BANDWIDTH LIFECYCLE SUMMARY


,bandwidth,rho_mean,rho_std,lambda_mean,lambda_std,unique_targets,target_count_mean,target_count_std,close_mean,reopen_mean,switch_mean,switch_std,actual_ratio_mean,actual_ratio_std,saved_ratio_mean,saved_ratio_std,semantic_keep_mean,semantic_keep_std,active_ratio_mean
0,bw0p1Mbps,0.5565,0.0354,0.9173,0.0030,7,5.0000,0.5000,1.1111,0.6667,0.1975,0.1335,0.6129,0.0514,0.3871,0.0514,0.8537,0.1249,0.5325
1,bw0p5Mbps,0.6181,0.0551,0.9124,0.0015,7,5.3333,0.5000,1.1111,0.6667,0.1975,0.1549,0.6626,0.0544,0.3374,0.0544,0.8930,0.0699,0.5695
2,bw1Mbps,0.6020,0.0297,0.9131,0.0015,7,5.3333,0.5000,1.1111,0.6667,0.1975,0.1549,0.6626,0.0544,0.3374,0.0544,0.8930,0.0699,0.5695
3,bw2Mbps,0.6490,0.0503,0.9133,0.0021,7,5.7778,0.6667,0.8889,0.4444,0.1481,0.1361,0.7217,0.0916,0.2783,0.0916,0.9281,0.0383,0.6288
4,bw5Mbps,0.6653,0.0444,0.9156,0.0022,7,5.8889,0.6009,0.8889,0.4444,0.1481,0.1111,0.7248,0.0662,0.2752,0.0662,0.9180,0.0593,0.6379



TARGET-SET FREQUENCY


,bandwidth,target,count
5,bw0p1Mbps,"(1, 3, 4, 6, 7)",2
6,bw0p1Mbps,"(2, 3, 4, 5, 6)",2
0,bw0p1Mbps,"(0, 1, 2, 3)",1
1,bw0p1Mbps,"(0, 1, 3, 6, 7)",1
2,bw0p1Mbps,"(0, 2, 3, 4, 6)",1
3,bw0p1Mbps,"(0, 3, 4, 5, 6)",1
4,bw0p1Mbps,"(1, 3, 4, 5, 6, 7)",1
12,bw0p5Mbps,"(1, 3, 4, 6, 7)",2
13,bw0p5Mbps,"(2, 3, 4, 5, 6)",2
7,bw0p5Mbps,"(0, 1, 2, 3, 4)",1



SANITY CHECK
bw0p1Mbps  | rho=0.5565±0.0354 | lambda=0.9173±0.0030 | unique_targets= 7 | target_n=5.00 | saved=0.3871 | switch=0.1975
bw0p5Mbps  | rho=0.6181±0.0551 | lambda=0.9124±0.0015 | unique_targets= 7 | target_n=5.33 | saved=0.3374 | switch=0.1975
bw1Mbps    | rho=0.6020±0.0297 | lambda=0.9131±0.0015 | unique_targets= 7 | target_n=5.33 | saved=0.3374 | switch=0.1975
bw2Mbps    | rho=0.6490±0.0503 | lambda=0.9133±0.0021 | unique_targets= 7 | target_n=5.78 | saved=0.2783 | switch=0.1481
bw5Mbps    | rho=0.6653±0.0444 | lambda=0.9156±0.0022 | unique_targets= 7 | target_n=5.89 | saved=0.2752 | switch=0.1481


In [34]:
# ============================================================
# Offline PPO vs Full Active
#
# Full Active:
#   RAW measured timestamps + RAW playback behavior
#
# PPO:
#   Counterfactual timestamps + PlaybackBuffer-style stall
# ============================================================

import numpy as np
import pandas as pd
import torch

EPS = 1e-8


# ============================================================
# Helpers
# ============================================================

def mean_or_nan(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    return float(x.mean()) if len(x) else np.nan


def p95_or_nan(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    return float(np.percentile(x, 95)) if len(x) else np.nan


def frame_time(run, key, fid):
    x = np.asarray(run[key][fid], dtype=np.float64)
    x = x[np.isfinite(x) & (x > 0)]
    return float(np.median(x)) if len(x) else np.nan


# ============================================================
# Lifecycle
# ============================================================

def build_active_history(run, state, pending, target, start, end):
    nsp = int(run["num_subpics"])
    idrs = np.asarray(run["idr_frames"], dtype=np.int64)
    closes = reopens = 0

    for sid in range(nsp):
        want = sid in target

        if state[sid] == ACTIVE and not want:
            state[sid] = CLOSED
            pending[sid] = -1
            closes += 1

        elif state[sid] == CLOSED and want:
            future = idrs[idrs >= start]
            if len(future):
                state[sid] = PENDING_REOPEN
                pending[sid] = int(future[0])

        elif state[sid] == PENDING_REOPEN and not want:
            state[sid] = CLOSED
            pending[sid] = -1

    active_history = []

    for fid in range(start, end + 1):
        for sid in range(nsp):
            if state[sid] == PENDING_REOPEN and pending[sid] == fid:
                state[sid] = ACTIVE
                pending[sid] = -1
                reopens += 1

        active_history.append((state == ACTIVE).copy())

    return active_history, closes, reopens


# ============================================================
# Traffic / Semantic Statistics
# ============================================================

def window_stats(run, start, end, active_history):
    total_bytes = sent_bytes = 0.0
    total_roi = kept_roi = 0.0

    for k, fid in enumerate(range(start, end + 1)):
        active = np.asarray(active_history[k], dtype=bool)

        B = np.asarray(run["frame_bytes"][fid], dtype=np.float64)
        R = np.asarray(run["roi"][fid], dtype=np.float64)

        total_bytes += float(B.sum())
        sent_bytes += float(B[active].sum())

        total_roi += float(R.sum())
        kept_roi += float(R[active].sum())

    actual_ratio = sent_bytes / max(total_bytes, EPS)

    return {
        "actual_ratio": float(actual_ratio),
        "saved_ratio": float(1.0 - actual_ratio),
        "semantic_keep": float(
            kept_roi / max(total_roi, EPS)
            if total_roi > EPS else 1.0
        ),
    }


# ============================================================
# Full Active RAW Playback State
#
# lag = actual_play - scheduled_play
#
# New stall generated by frame f:
#   max(0, lag_f - lag_{f-1})
#
# This avoids counting one stall repeatedly on later frames.
# ============================================================

class RawPlaybackState:
    def __init__(self):
        self.prev_lag_ns = None


def raw_full_active_qos(run, start, end, playback_state):
    source = np.asarray(run["frame_source_time_ns"], dtype=np.float64)
    recv = np.asarray(run["receive_complete_ns"], dtype=np.float64)
    decode = np.asarray(run["decode_end_ns"], dtype=np.float64)

    delivery, e2e = [], []
    stall, miss, missing = [], [], []

    for fid in range(start, end + 1):
        r = np.asarray(recv[fid], dtype=np.float64)
        d = np.asarray(decode[fid], dtype=np.float64)

        valid = (
            np.isfinite(r) & (r > 0) &
            np.isfinite(d) & (d > 0)
        )

        if not valid.all():
            missing.append(1.0)
            continue

        missing.append(0.0)

        src = float(source[fid])
        last_recv = float(r.max())
        decode_end = float(d.max())

        # Source -> all SPs received
        delivery.append((last_recv - src) / 1e6)

        # Source -> all SP decode calls complete
        e2e.append((decode_end - src) / 1e6)

        scheduled = frame_time(run, "scheduled_play_ns", fid)
        actual = frame_time(run, "actual_play_ns", fid)

        if not np.isfinite(scheduled) or not np.isfinite(actual):
            continue

        lag_ns = max(0.0, actual - scheduled)

        if playback_state.prev_lag_ns is None:
            stall_ns = lag_ns
        else:
            stall_ns = max(
                0.0,
                lag_ns - playback_state.prev_lag_ns
            )

        playback_state.prev_lag_ns = lag_ns

        stall.append(stall_ns / 1e6)
        miss.append(float(stall_ns > 0))

    stall_events = [float(x > EPS) for x in stall]

    return {
        "delivery_latency_ms": mean_or_nan(delivery),
        "delivery_p95_ms": p95_or_nan(delivery),

        "e2e_latency_ms": mean_or_nan(e2e),
        "e2e_p95_ms": p95_or_nan(e2e),

        "deadline_miss": mean_or_nan(miss),

        "stall_ratio": mean_or_nan(stall_events),
        "stall_ms": mean_or_nan(stall),
        "stall_p95_ms": p95_or_nan(stall),
        "total_stall_ms": float(np.sum(stall)) if len(stall) else np.nan,

        "missing_ratio": mean_or_nan(missing),
    }


# ============================================================
# PPO Counterfactual Estimator
#
# Timestamp estimation logic unchanged.
# Stall follows PlaybackBuffer behavior:
#
# due -> frame ready?
#   yes: play at due, no stall
#   no : wait until ready, stall = ready - due
#
# after play/recovery:
#   next_due = actual_play + 1/FPS
# ============================================================

class PPOCounterfactualEstimator:
    def __init__(self, run):
        self.run = run
        self.nsp = int(run["num_subpics"])

        self.source = np.asarray(
            run["frame_source_time_ns"],
            dtype=np.float64
        )

        self.send = np.asarray(
            run["subpic_send_time_ns"],
            dtype=np.float64
        )

        self.recv_first = np.asarray(
            run["receive_first_ns"],
            dtype=np.float64
        )

        self.recv = np.asarray(
            run["receive_complete_ns"],
            dtype=np.float64
        )

        self.decode = np.asarray(
            run["decode_end_ns"],
            dtype=np.float64
        )

        self.frame_bytes = np.asarray(
            run["frame_bytes"],
            dtype=np.float64
        )

        self.carry_shift_ns = 0.0

        # Playback state persists across 1-s control units
        self.interval_ns = 1e9 / float(run["fps"])
        self.next_play_ns = None

    # --------------------------------------------------------
    # Effective receiver-side rate
    # --------------------------------------------------------

    def estimate_rate(self, start, end):
        rf = self.recv_first[start:end + 1].reshape(-1)
        rc = self.recv[start:end + 1].reshape(-1)
        B = self.frame_bytes[start:end + 1].reshape(-1)

        valid = (
            np.isfinite(rf) & (rf > 0) &
            np.isfinite(rc) & (rc > 0) &
            np.isfinite(B) & (B > 0)
        )

        if not valid.any():
            return np.nan

        t0 = float(rf[valid].min())
        t1 = float(rc[valid].max())
        duration_s = (t1 - t0) / 1e9

        if duration_s <= 0:
            return np.nan

        return float(B[valid].sum() * 8.0 / duration_s)

    # --------------------------------------------------------
    # Initialize playback state at first controlled frame.
    #
    # Previous 1-s unit is still raw Full Active, so use the
    # previous frame's actual playback time as boundary state.
    # --------------------------------------------------------

    def init_playback_clock(self, start):
        if self.next_play_ns is not None:
            return

        prev_fid = start - 1

        if prev_fid >= 0:
            prev_actual = frame_time(
                self.run,
                "actual_play_ns",
                prev_fid
            )

            if np.isfinite(prev_actual):
                self.next_play_ns = (
                    prev_actual
                    + self.interval_ns
                )
                return

        scheduled = frame_time(
            self.run,
            "scheduled_play_ns",
            start
        )

        if np.isfinite(scheduled):
            self.next_play_ns = scheduled

    # --------------------------------------------------------
    # One 1-second control unit
    # --------------------------------------------------------

    def simulate_window(self, start, end, active_history):
        rate_bps = self.estimate_rate(start, end)

        if not np.isfinite(rate_bps) or rate_bps <= 0:
            raise RuntimeError(
                f"Invalid effective rate: "
                f"frames={start}-{end}, rate={rate_bps}"
            )

        self.init_playback_clock(start)

        active_map = {
            fid: np.asarray(active_history[k], dtype=bool)
            for k, fid in enumerate(range(start, end + 1))
        }

        events = []

        for fid in range(start, end + 1):
            for sid in range(self.nsp):
                t = float(self.send[fid, sid])

                if np.isfinite(t) and t > 0:
                    events.append((t, fid, sid))

        events.sort(key=lambda x: x[0])

        new_send = {}
        new_recv = {}
        new_decode = {}

        shift_ns = self.carry_shift_ns

        # ----------------------------------------------------
        # Counterfactual transmission timeline
        # ----------------------------------------------------

        for _, fid, sid in events:
            active = active_map[fid][sid]

            orig_send = float(self.send[fid, sid])
            orig_recv = float(self.recv[fid, sid])
            orig_decode = float(self.decode[fid, sid])
            src = float(self.source[fid])

            # CLOSED / PENDING:
            # remove transmission occupancy
            if not active:
                B = float(self.frame_bytes[fid, sid])

                shift_ns += (
                    B * 8.0 / rate_bps * 1e9
                )

                continue

            max_shift = max(
                0.0,
                orig_send - src
            )

            actual_shift = min(
                shift_ns,
                max_shift
            )

            new_s = orig_send - actual_shift

            # Idle / frame-release boundary absorbs excess shift
            shift_ns = actual_shift

            key = (fid, sid)

            new_send[key] = new_s

            if np.isfinite(orig_recv) and orig_recv > 0:
                new_recv[key] = (
                    orig_recv - actual_shift
                )

            if np.isfinite(orig_decode) and orig_decode > 0:
                new_decode[key] = (
                    orig_decode - actual_shift
                )

        self.carry_shift_ns = shift_ns

        # ----------------------------------------------------
        # QoS + PlaybackBuffer-style stall
        # ----------------------------------------------------

        delivery, e2e = [], []
        stall, miss, missing = [], [], []

        for fid in range(start, end + 1):
            active_ids = np.flatnonzero(
                active_map[fid]
            )

            if not len(active_ids):
                continue

            keys = [
                (fid, sid)
                for sid in active_ids
            ]

            complete = all(
                k in new_recv
                and k in new_decode
                for k in keys
            )

            if not complete:
                missing.append(1.0)
                continue

            missing.append(0.0)

            recvs = [
                new_recv[k]
                for k in keys
            ]

            decodes = [
                new_decode[k]
                for k in keys
            ]

            src = float(self.source[fid])

            # Playback frame-ready = all required ACTIVE SPs received
            ready_ns = max(recvs)

            # Decode completion kept as separate E2E metric
            decode_end_ns = max(decodes)

            delivery.append(
                (ready_ns - src) / 1e6
            )

            e2e.append(
                (decode_end_ns - src) / 1e6
            )

            # ------------------------------------------------
            # Real playback-buffer logic
            #
            # If frame is ready before due:
            #     no stall, play at due
            #
            # If frame is not ready at due:
            #     wait until ready
            #     stall = ready - due
            #
            # Playback clock resumes from actual play time.
            # ------------------------------------------------

            if self.next_play_ns is None:
                scheduled = frame_time(
                    self.run,
                    "scheduled_play_ns",
                    fid
                )

                if not np.isfinite(scheduled):
                    continue

                self.next_play_ns = scheduled

            due_ns = self.next_play_ns

            if ready_ns <= due_ns:
                actual_play_ns = due_ns
                stall_ns = 0.0
            else:
                actual_play_ns = ready_ns
                stall_ns = ready_ns - due_ns

            stall.append(
                stall_ns / 1e6
            )

            miss.append(
                float(stall_ns > 0)
            )

            # IMPORTANT:
            # resume playback clock from actual play / recovery
            self.next_play_ns = (
                actual_play_ns
                + self.interval_ns
            )

        stall_events = [
            float(x > EPS)
            for x in stall
        ]

        return {
            "delivery_latency_ms": mean_or_nan(delivery),
            "delivery_p95_ms": p95_or_nan(delivery),

            "e2e_latency_ms": mean_or_nan(e2e),
            "e2e_p95_ms": p95_or_nan(e2e),

            "deadline_miss": mean_or_nan(miss),

            "stall_ratio": mean_or_nan(stall_events),
            "stall_ms": mean_or_nan(stall),
            "stall_p95_ms": p95_or_nan(stall),
            "total_stall_ms": float(np.sum(stall)) if len(stall) else np.nan,

            "missing_ratio": mean_or_nan(missing),

            "estimated_rate_mbps": float(
                rate_bps / 1e6
            ),
        }


# ============================================================
# PPO Evaluation
# action[t] controls window[t+1]
# ============================================================

@torch.no_grad()
def evaluate_ppo(model, run_dict, runs):
    model.eval()
    rows = []

    for run_key, samples in run_dict.items():
        if len(samples) < 2:
            continue

        video_id, bandwidth = run_key
        run = runs[run_key]
        nsp = int(run["num_subpics"])

        state = np.full(
            nsp,
            ACTIVE,
            dtype=np.int8
        )

        pending = np.full(
            nsp,
            -1,
            dtype=np.int64
        )

        estimator = PPOCounterfactualEstimator(
            run
        )

        for t in range(len(samples) - 1):
            s = samples[t]
            nxt = samples[t + 1]

            obs = (
                build_sp_features(
                    s,
                    state
                ),
                normalize_global(s)
            )

            action, _, _ = sample_action(
                model,
                obs,
                deterministic=True
            )

            rho, lam = map(
                float,
                action
            )

            target = set(
                map(
                    int,
                    lifecycle_target(
                        s,
                        rho,
                        lam
                    )["selected"]
                )
            )

            start = int(
                nxt["start_fid"]
            )

            end = int(
                nxt["end_fid"]
            )

            active_history, closes, reopens = build_active_history(
                run,
                state,
                pending,
                target,
                start,
                end
            )

            stats = window_stats(
                run,
                start,
                end,
                active_history
            )

            switch_ratio = (
                closes + reopens
            ) / max(nsp, 1)

            pressure = float(
                np.clip(
                    max(
                        s["bif_cwnd_mean"],
                        s["bif_cwnd_last"]
                    ),
                    0,
                    1
                )
            )

            reward = (
                stats["semantic_keep"]
                - pressure * stats["actual_ratio"]
                - SWITCH_WEIGHT * switch_ratio
            )

            qos = estimator.simulate_window(
                start,
                end,
                active_history
            )

            rows.append({
                "method": "PPO Estimated",
                "video_id": video_id,
                "bandwidth": bandwidth,
                "control_idx": t,

                "rho": rho,
                "lambda": lam,

                "saved_ratio":
                    stats["saved_ratio"],

                "semantic_keep":
                    stats["semantic_keep"],

                "switch_ratio":
                    switch_ratio,

                "reward":
                    reward,

                **qos,
            })

    return pd.DataFrame(rows)


# ============================================================
# Full Active RAW Evaluation
#
# Playback state MUST persist across 1-s units.
# ============================================================

def evaluate_full_active_raw(run_dict, runs):
    rows = []

    for run_key, samples in run_dict.items():
        if len(samples) < 2:
            continue

        video_id, bandwidth = run_key
        run = runs[run_key]

        playback_state = RawPlaybackState()

        # ----------------------------------------------------
        # Warm up stall state using the first uncontrolled unit
        # ----------------------------------------------------

        first = samples[0]

        raw_full_active_qos(
            run,
            int(first["start_fid"]),
            int(first["end_fid"]),
            playback_state
        )

        # ----------------------------------------------------
        # Same controlled units used for PPO comparison
        # ----------------------------------------------------

        for t in range(len(samples) - 1):
            s = samples[t]
            nxt = samples[t + 1]

            start = int(
                nxt["start_fid"]
            )

            end = int(
                nxt["end_fid"]
            )

            pressure = float(
                np.clip(
                    max(
                        s["bif_cwnd_mean"],
                        s["bif_cwnd_last"]
                    ),
                    0,
                    1
                )
            )

            qos = raw_full_active_qos(
                run,
                start,
                end,
                playback_state
            )

            rows.append({
                "method": "Full Active Raw",
                "video_id": video_id,
                "bandwidth": bandwidth,
                "control_idx": t,

                "rho": 1.0,
                "lambda": np.nan,

                "saved_ratio": 0.0,
                "semantic_keep": 1.0,
                "switch_ratio": 0.0,

                "reward": 1.0 - pressure,

                **qos,
            })

    return pd.DataFrame(rows)


# ============================================================
# Validation
# ============================================================

val_ppo = evaluate_ppo(
    model,
    val_runs,
    runs
)

val_full = evaluate_full_active_raw(
    val_runs,
    runs
)

compare = pd.concat(
    [val_full, val_ppo],
    ignore_index=True
)


# ============================================================
# Compact Final Table
# ============================================================

final_table = (
    compare
    .groupby(
        [
            "method",
            "video_id",
            "bandwidth"
        ],
        dropna=False
    )
    .agg(
        saved_ratio=(
            "saved_ratio",
            "mean"
        ),

        semantic_keep=(
            "semantic_keep",
            "mean"
        ),

        delivery_latency_ms=(
            "delivery_latency_ms",
            "mean"
        ),

        e2e_latency_ms=(
            "e2e_latency_ms",
            "mean"
        ),

        deadline_miss=(
            "deadline_miss",
            "mean"
        ),

        stall_ratio=(
            "stall_ratio",
            "mean"
        ),

        mean_stall_ms=(
            "stall_ms",
            "mean"
        ),

        total_stall_ms=(
            "total_stall_ms",
            "sum"
        ),

        switch_ratio=(
            "switch_ratio",
            "mean"
        ),

        missing_ratio=(
            "missing_ratio",
            "mean"
        ),
    )
    .reset_index()
)

print("\n" + "=" * 120)
print("FULL ACTIVE RAW vs PPO COUNTERFACTUAL")
print("=" * 120)

display(
    final_table.round(4)
)


FULL ACTIVE RAW vs PPO COUNTERFACTUAL


,method,video_id,bandwidth,saved_ratio,semantic_keep,delivery_latency_ms,e2e_latency_ms,deadline_miss,stall_ratio,mean_stall_ms,total_stall_ms,switch_ratio,missing_ratio
0,Full Active Raw,BQTerrace_QP32_3x3,bw0p1Mbps,0.0000,1.0000,189860.7619,190006.4106,0.4389,0.4389,415.9936,224636.5565,0.0000,0.0
1,Full Active Raw,BQTerrace_QP32_3x3,bw0p5Mbps,0.0000,1.0000,32206.8941,32572.8968,0.4463,0.4463,78.9987,42659.2730,0.0000,0.0
2,Full Active Raw,BQTerrace_QP32_3x3,bw1Mbps,0.0000,1.0000,11075.0473,11521.0827,0.4500,0.4500,26.4025,14257.3404,0.0000,0.0
3,Full Active Raw,BQTerrace_QP32_3x3,bw2Mbps,0.0000,1.0000,236.3686,1050.6014,0.4259,0.4259,2.7306,1474.5316,0.0000,0.0
4,Full Active Raw,BQTerrace_QP32_3x3,bw5Mbps,0.0000,1.0000,46.5554,522.5050,0.4556,0.4556,2.4675,1332.4572,0.0000,0.0
5,PPO Estimated,BQTerrace_QP32_3x3,bw0p1Mbps,0.3871,0.8537,124185.9803,125522.2332,0.1037,0.1037,266.8740,144111.9764,0.1975,0.0
6,PPO Estimated,BQTerrace_QP32_3x3,bw0p5Mbps,0.3374,0.8930,19014.6507,19534.7711,0.0648,0.0648,55.4949,29967.2568,0.1975,0.0
7,PPO Estimated,BQTerrace_QP32_3x3,bw1Mbps,0.3374,0.8930,4198.2772,4706.5261,0.0463,0.0463,20.3978,11014.8088,0.1975,0.0
8,PPO Estimated,BQTerrace_QP32_3x3,bw2Mbps,0.2783,0.9281,204.6468,1038.5773,0.2537,0.2537,14.9274,8060.8187,0.1481,0.0
9,PPO Estimated,BQTerrace_QP32_3x3,bw5Mbps,0.2752,0.9180,41.6397,515.8923,0.7278,0.7278,15.7819,8522.2121,0.1481,0.0


In [24]:
import numpy as np
import pandas as pd

EPS = 1e-8


# ============================================================
# 1-s L1 normalization
# ============================================================

def normalize_unit(x):
    x = np.asarray(x, dtype=np.float32)
    s = float(x.sum())
    return np.zeros_like(x) if s <= EPS else x / s


# ============================================================
# Score-aware near-budget selector
# O(N log N), no enumeration
# ============================================================

def select_near_budget(score, frame_bytes, target_bytes):

    score = np.asarray(score, dtype=np.float32)
    frame_bytes = np.asarray(frame_bytes, dtype=np.float32)

    order = np.argsort(
        -score,
        kind="stable"
    )

    selected = []
    used_bytes = 0.0

    for sid in order:

        sid = int(sid)

        selected.append(sid)

        used_bytes += float(
            frame_bytes[sid]
        )

        # Allow the first SP that reaches/exceeds target
        if used_bytes >= target_bytes:
            break

    # At least one SP
    if len(selected) == 0 and len(order) > 0:
        sid = int(order[0])
        selected = [sid]
        used_bytes = float(
            frame_bytes[sid]
        )

    return (
        selected,
        float(used_bytes),
        order
    )


# ============================================================
# Lifecycle selection
#
# rho:
#   target transmission ratio
#
# lambda_:
#   ROI vs change preference
# ============================================================

def lifecycle_select(sample, rho, lambda_):

    rho = float(np.clip(rho, 0.0, 1.0))
    lambda_ = float(np.clip(lambda_, 0.0, 1.0))

    # --------------------------------------------------------
    # R / T
    # --------------------------------------------------------

    R = np.asarray(
        sample["roi_mean"],
        dtype=np.float32
    )

    T = np.asarray(
        sample["change_mean"],
        dtype=np.float32
    )

    frame_bytes = np.asarray(
        sample["frame_bytes"],
        dtype=np.float32
    )

    assert R.shape == T.shape == frame_bytes.shape

    if np.any(R < 0):
        raise ValueError("ROI contains negative values.")

    if np.any(T < 0):
        raise ValueError("Change contains negative values.")

    # --------------------------------------------------------
    # 1-second relative normalization
    # --------------------------------------------------------

    R_hat = normalize_unit(R)
    T_hat = normalize_unit(T)

    # --------------------------------------------------------
    # Lifecycle score
    # --------------------------------------------------------

    score = (
        lambda_ * R_hat
        +
        (1.0 - lambda_) * T_hat
    ).astype(np.float32)

    # --------------------------------------------------------
    # Target transmission load
    # --------------------------------------------------------

    total_bytes = float(
        frame_bytes.sum()
    )

    target_bytes = (
        rho * total_bytes
    )

    # --------------------------------------------------------
    # Selection
    # --------------------------------------------------------

    selected, actual_bytes, order = (
        select_near_budget(
            score,
            frame_bytes,
            target_bytes,
        )
    )

    selected_set = set(selected)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    actual_ratio = (
        actual_bytes
        / max(total_bytes, EPS)
    )

    budget_error_bytes = (
        actual_bytes
        - target_bytes
    )

    budget_error_ratio = (
        budget_error_bytes
        / max(total_bytes, EPS)
    )

    rank_map = {
        int(sid): rank + 1
        for rank, sid in enumerate(order)
    }

    # --------------------------------------------------------
    # Detail table
    # --------------------------------------------------------

    detail = pd.DataFrame([
        {
            "sp": sid,
            "R_roi_mean": float(R[sid]),
            "T_change_mean": float(T[sid]),
            "R_hat": float(R_hat[sid]),
            "T_hat": float(T_hat[sid]),
            "score": float(score[sid]),
            "bytes": int(frame_bytes[sid]),
            "score_per_byte": float(
                score[sid]
                / max(frame_bytes[sid], EPS)
            ),
            "rank": rank_map[sid],
            "selected": sid in selected_set,
        }
        for sid in range(len(R))
    ]).sort_values("rank").reset_index(drop=True)

    return {
        "rho": rho,
        "lambda": lambda_,

        "R": R,
        "T": T,
        "R_hat": R_hat,
        "T_hat": T_hat,
        "score": score,

        "R_total": float(R.sum()),
        "T_total": float(T.sum()),

        "total_bytes": total_bytes,
        "target_bytes": target_bytes,
        "actual_bytes": actual_bytes,

        "target_ratio": rho,
        "actual_ratio": actual_ratio,

        "budget_error_bytes": budget_error_bytes,
        "budget_error_ratio": budget_error_ratio,

        "selected": selected,
        "num_selected": len(selected),
        "num_subpics": len(R),

        "detail": detail,
    }


# ============================================================
# Test one sample
# ============================================================

sample = all_samples[0]

result = lifecycle_select(
    sample,
    rho=0.50,
    lambda_=0.70,
)

print("=" * 70)
print("Lifecycle Selection")
print("=" * 70)

print("video         :", sample["video_id"])
print("network       :", sample["bandwidth"])
print("control       :", sample["control_idx"])
print()

print(f"rho           : {result['rho']:.3f}")
print(f"lambda        : {result['lambda']:.3f}")

print(
    f"bytes         : "
    f"target={result['target_bytes']:.0f}, "
    f"actual={result['actual_bytes']:.0f}, "
    f"total={result['total_bytes']:.0f}"
)

print(
    f"ratio         : "
    f"target={result['target_ratio']:.3f}, "
    f"actual={result['actual_ratio']:.3f}"
)

print(
    f"budget error  : "
    f"{result['budget_error_ratio']:+.3f}"
)

print("selected      :", result["selected"])

display(result["detail"])


# ============================================================
# Sensitivity test
# ============================================================

rows = []

for rho in [0.25, 0.50, 0.75, 1.00]:
    for lambda_ in [0.00, 0.25, 0.50, 0.75, 1.00]:

        r = lifecycle_select(
            sample,
            rho=rho,
            lambda_=lambda_,
        )

        rows.append({
            "rho": rho,
            "lambda": lambda_,
            "selected": str(r["selected"]),
            "n_selected": r["num_selected"],
            "actual_ratio": r["actual_ratio"],
            "budget_error_ratio":
                r["budget_error_ratio"],
        })

display(pd.DataFrame(rows))

Lifecycle Selection
video         : BasketballDrive_QP32_3x3
network       : bw0p1Mbps
control       : 0

rho           : 0.500
lambda        : 0.700
bytes         : target=168841, actual=196373, total=337682
ratio         : target=0.500, actual=0.582
budget error  : +0.082
selected      : [3, 6, 5]


,sp,R_roi_mean,T_change_mean,R_hat,T_hat,score,bytes,score_per_byte,rank,selected
0,3,0.550863,0.112032,0.393102,0.136544,0.316135,99193,3.187068e-06,1,True
1,6,0.281759,0.103034,0.201066,0.125577,0.178419,52476,3.400020e-06,2,True
2,5,0.240212,0.109229,0.171418,0.133128,0.159931,44704,3.577555e-06,3,True
3,4,0.184176,0.127279,0.131430,0.155128,0.138539,44767,3.094673e-06,4,False
4,0,0.102734,0.103799,0.073312,0.126511,0.089272,33979,2.627262e-06,5,False
5,8,0.022046,0.123571,0.015732,0.150608,0.056195,14098,3.986018e-06,6,False
6,7,0.018600,0.117159,0.013273,0.142793,0.052129,13482,3.866569e-06,7,False
7,1,0.000909,0.019436,0.000648,0.023689,0.007561,17953,4.211313e-07,8,False
8,2,0.000025,0.004941,0.000018,0.006022,0.001819,17030,1.068186e-07,9,False


,rho,lambda,selected,n_selected,actual_ratio,budget_error_ratio
0,0.25,0.00,"[4, 8, 7, 3]",4,0.507993,0.257993
1,0.25,0.25,[3],1,0.293747,0.043747
2,0.25,0.50,[3],1,0.293747,0.043747
3,0.25,0.75,[3],1,0.293747,0.043747
4,0.25,1.00,[3],1,0.293747,0.043747
5,0.50,0.00,"[4, 8, 7, 3]",4,0.507993,0.007993
6,0.50,0.25,"[3, 4, 6]",3,0.581719,0.081719
7,0.50,0.50,"[3, 6, 5]",3,0.581532,0.081532
8,0.50,0.75,"[3, 6, 5]",3,0.581532,0.081532
9,0.50,1.00,"[3, 6, 5]",3,0.581532,0.081532
